<a href="https://colab.research.google.com/github/r-karra/Gemma-2-9B-JEE-Socratic/blob/main/test_Gemma_2_9B_JEE_Socratic_Final_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import userdata
import os
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Retrieve the token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Use the token variable directly
model_id = "r-karra/Gemma-2-9B-JEE-Socratic-Final"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=hf_token,
    device_map="auto",
    torch_dtype="auto" # 'auto' automatically selects the best dtype (fp16/bf16)
)

print("Model loaded! You are ready to test.")

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Model loaded! You are ready to test.


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from google.colab import userdata

# 1. Configuration
model_id = "r-karra/Gemma-2-9B-JEE-Socratic-Final"
token = userdata.get('HF_TOKEN')

# 2. Load tokenizer and FORCIBLY set the chat template to the Gemma 2 default
# This bypasses the broken 'tokenizer_config.json' from the Unsloth save
print("Loading base tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-9b-it", token=token)

print("Loading fine-tuned model weights...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=token,
    device_map="auto",
    torch_dtype=torch.float16
)

def ask_tutor(question):
    # Manually define the messages
    messages = [
        {"role": "user", "content": f"You are an expert IIT-JEE Advanced tutor. Use the Socratic method.\n\nQuestion: {question}"}
    ]

    # Use the tokenizer directly without apply_chat_template if it keeps failing
    # Or use the chat template from the base tokenizer we just loaded
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(inputs, max_new_tokens=500)

    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

# Test run
print(ask_tutor("How can I determine if a reaction is spontaneous using Gibbs free energy?"))

Loading base tokenizer...
Loading fine-tuned model weights...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

AttributeError: 